## 1. Setup

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from pathlib import Path
from sklearn.model_selection import train_test_split

root = Path.cwd().parents[0]

data_path = root / "data" / "student_mental_health.csv"

df = pd.read_csv(data_path)

target = "Mental_Health_Risk"

## 2. Numerical and categorical columns

In [4]:
train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df[target]
)

X_train = train_df.drop(columns=["Student_ID", target])
y_train = train_df[target].copy()

X_test = test_df.drop(columns=["Student_ID", target])
y_test = test_df[target].copy()

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
binary_features = ["Part_Time_Job", "Previous_Mental_Health_History"]
ordinal_features = ["Physical_Activity", "Diet_Quality"]
nominal_features = ["Gender"]

print("Numerical:", numeric_features)

Numerical: ['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']


In [5]:
Q1 = X_train[numeric_features].quantile(0.25)
Q3 = X_train[numeric_features].quantile(0.75)

IQR = Q3 - Q1

lower_bounds = Q1 - 1.5 * IQR
upper_bounds = Q3 + 1.5 * IQR

outlier_flags = (X_train[numeric_features] < lower_bounds) | (
    X_train[numeric_features] > upper_bounds
)

flagged_values = X_train[numeric_features].where(outlier_flags)

outlier_review = pd.DataFrame(
    {
        "IQR_lower": lower_bounds,
        "IQR_upper": upper_bounds,
        "Flagged_count": outlier_flags.sum(),
        "Flagged_min": flagged_values.min(),
        "Flagged_max": flagged_values.max(),
    }
)

outlier_review = outlier_review[outlier_review["Flagged_count"] > 0]

display(outlier_review.round(2))


column = "Study_Hours_Per_Week"

display(train_df.loc[outlier_flags[column], ["Student_ID", column]].sort_values(column))

,IQR_lower,IQR_upper,Flagged_count,Flagged_min,Flagged_max
Age,15.50,27.50,2,28.0,28.00
Sleep_Hours,3.75,9.75,9,3.6,10.00
Study_Hours_Per_Week,4.00,36.00,7,2.4,38.30
GPA,1.60,3.52,5,1.5,3.66
Social_Media_Hours,3.05,9.05,10,2.1,10.00
Screen_Time_Hours,3.95,13.15,7,3.3,14.00
Attendance_Rate,65.35,104.95,10,54.4,65.30


,Student_ID,Study_Hours_Per_Week
866,STU0867,2.4
1042,STU1043,2.7
453,STU0454,3.2
519,STU0520,3.6
340,STU0341,3.9
937,STU0938,37.5
716,STU0717,38.3


## 4. Preprocessing pipeline

In [ ]:
numeric_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),

    ("encoder", OrdinalEncoder(
        categories=[
            ["No", "Yes"],
            ["No", "Yes"] 
        ]
    ))
])

ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),

    ("encoder", OrdinalEncoder(
        categories=[
            ["Low", "Moderate", "High"],  
            ["Poor", "Average", "Good"]  
        ]
    )),

    ("scaler", StandardScaler())
])

nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),

    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("binary", binary_pipeline, binary_features),
    ("ordinal", ordinal_pipeline, ordinal_features),
    ("nominal", nominal_pipeline, nominal_features)
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)


prepared_df = pd.DataFrame(
    X_train_prepared, columns=preprocessor.get_feature_names_out(), index=X_train.index
)

display(prepared_df.head())

print("Training shape:", X_train_prepared.shape)
print("Test shape:", X_test_prepared.shape)
print("Missing values:", prepared_df.isna().sum().sum())

,num__Age,num__Year_of_Study,num__Sleep_Hours,num__Study_Hours_Per_Week,num__GPA,num__Social_Media_Hours,num__Screen_Time_Hours,num__Financial_Stress,num__Family_Support,num__Attendance_Rate,binary__Part_Time_Job,binary__Previous_Mental_Health_History,ordinal__Physical_Activity,ordinal__Diet_Quality,nominal__Gender_Female,nominal__Gender_Male,nominal__Gender_Non-binary
726,-0.063706,-0.471636,-1.404182,-0.238603,0.074214,0.148597,1.098992,0.713748,0.643008,-1.244652,0.0,0.0,0.191167,0.080487,1.0,0.0,0.0
1384,0.875744,-1.232340,-1.676036,-1.330997,-0.976093,0.327697,0.194658,-0.842691,0.368687,-0.465696,0.0,0.0,1.581473,0.080487,1.0,0.0,0.0
1579,-1.003157,-0.471636,0.408177,-0.516037,-1.034443,-0.299153,-0.830254,-0.305988,0.752736,0.772288,0.0,0.0,0.191167,0.080487,1.0,0.0,0.0
1098,0.875744,-1.232340,1.404975,-0.082547,0.891120,0.775447,1.098992,0.821088,-0.948052,-0.215317,0.0,0.0,0.191167,0.080487,1.0,0.0,0.0
582,0.406019,0.289067,0.408177,0.663055,-0.275888,-0.388703,-0.347943,0.660077,0.697872,-1.453301,1.0,1.0,1.581473,0.080487,1.0,0.0,0.0


Training shape: (1600, 17)
Test shape: (400, 17)
Missing values: 0


In [ ]:
class_balance = pd.DataFrame(
    {
        "Count": y_train.value_counts(),
        "Percent": y_train.value_counts(normalize=True) * 100,
    }
)

display(class_balance)